## 基于 LLM 的 LAS 曲线自动归类与增量导出

目标：

- 遍历 `data/all_well_las_vp_only_head` 下每个 LAS 文件。
- 每个文件调用 1 次 LLM（失败最多重试 1 次）。
- 每个文件落地 1 个 JSON 结果。
- 以 `data/template_import.csv` 的列为模板，增量写入 `data/import.csv`。
- `import.csv` 使用 `utf-8-sig` 编码，便于 Excel 直接打开。

该工作流很不成熟。


In [ ]:
from __future__ import annotations

import json
import time
from pathlib import Path
from typing import Any, Dict, List

import pandas as pd
import requests

# 自动定位项目根目录
ROOT = Path.cwd().resolve()
if not (ROOT / "data").exists() and (ROOT.parent / "data").exists():
    ROOT = ROOT.parent

DATA_DIR = ROOT / "data"
LAS_DIR = DATA_DIR / "all_well_las_vp_only_head"
TEMPLATE_CSV = DATA_DIR / "template_import.csv"
OUT_CSV = DATA_DIR / "import.csv"
JSON_OUT_DIR = DATA_DIR / "llm_curve_json"
API_KEY_FILE = DATA_DIR / "LLM" / ".ARK_API_KEY"
MODEL_NAME_FILE = DATA_DIR / "LLM" / ".MODEL_NAME"

API_URL = "https://ark.cn-beijing.volces.com/api/v3/chat/completions"
MAX_RETRY = 1  # 失败后最多再试 1 次

JSON_OUT_DIR.mkdir(parents=True, exist_ok=True)

if not TEMPLATE_CSV.exists():
    raise FileNotFoundError(f"模板 CSV 不存在: {TEMPLATE_CSV}")
if not LAS_DIR.exists():
    raise FileNotFoundError(f"LAS 目录不存在: {LAS_DIR}")
if not API_KEY_FILE.exists():
    raise FileNotFoundError(f"API Key 文件不存在: {API_KEY_FILE}")
if not MODEL_NAME_FILE.exists():
    raise FileNotFoundError(f"模型名文件不存在: {MODEL_NAME_FILE}")

api_key = API_KEY_FILE.read_text(encoding="utf-8").strip()
if not api_key:
    raise ValueError(f"API Key 文件为空: {API_KEY_FILE}")

model_name = MODEL_NAME_FILE.read_text(encoding="utf-8").strip()
if not model_name:
    raise ValueError(f"模型名文件为空: {MODEL_NAME_FILE}")

template_df = pd.read_csv(TEMPLATE_CSV, encoding="utf-8")
columns = template_df.columns.tolist()
if not columns:
    raise ValueError("模板 CSV 没有列")

KEY_COLUMN = columns[0]  # 现在模板首列是“文件名”

print(f"ROOT: {ROOT}")
print(f"模板列数: {len(columns)}")
print(f"主键列: {KEY_COLUMN}")
print(f"LAS 文件数: {len(list(LAS_DIR.glob('*.las')))}")
print(f"模型: {model_name}")

In [ ]:
def normalize_value(v: Any) -> str:
    if v is None:
        return ""
    if isinstance(v, list):
        return ", ".join(str(x).strip() for x in v if str(x).strip())
    if isinstance(v, str):
        return v.strip()
    return str(v).strip()


def build_prompt(file_name: str, las_text: str, schema_columns: List[str]) -> str:
    # 文件名由调用端决定，不允许模型自行改写
    return (
        "我希望你帮我对测井曲线进行分类。我会给你一份 LAS 文件，请识别曲线并按目标字段分类，输出 JSON。\n"
        "要求：\n"
        "1) 只输出一个 JSON 对象，不要输出额外解释。\n"
        "2) JSON 的键必须严格使用给定字段名。\n"
        "3) 每个字段值用字符串，多个曲线用逗号+空格分隔。\n"
        "4) 如果某个字段为空，就填空字符串。\n"
        "5) 文件名字段必须等于输入文件名。\n\n"
        f"文件名: {file_name}\n"
        f"目标字段: {schema_columns}\n"
        "LAS 文件如下：\n"
        f"{las_text}\n\n"
        "请仅返回 JSON。"
    )


def call_llm_once(prompt: str) -> Dict[str, Any]:
    headers = {
        "Content-Type": "application/json",
        "Authorization": f"Bearer {api_key}",
    }
    payload = {
        "model": model_name,
        "messages": [
            {"role": "system", "content": "你是人工智能助手。"},
            {"role": "user", "content": prompt},
        ],
        "temperature": 0,
    }
    resp = requests.post(API_URL, headers=headers, json=payload, timeout=120)
    resp.raise_for_status()
    data = resp.json()

    content = data["choices"][0]["message"]["content"]
    # 兼容模型可能返回 ```json ... ```
    content = content.strip()
    if content.startswith("```"):
        content = content.strip("`")
        if content.lower().startswith("json"):
            content = content[4:].strip()
    return json.loads(content)


def validate_and_normalize_result(
    result: Dict[str, Any], file_name: str, schema_columns: List[str], key_column: str
) -> Dict[str, str]:
    if not isinstance(result, dict):
        raise ValueError("LLM 返回不是 JSON 对象")

    normalized: Dict[str, str] = {c: "" for c in schema_columns}
    for c in schema_columns:
        if c in result:
            normalized[c] = normalize_value(result[c])

    # 按你的要求：首列为文件名
    normalized[key_column] = file_name
    return normalized

In [ ]:
# 第一步：仅补全单井 JSON（已存在则跳过）
las_files = sorted(LAS_DIR.glob("*.las"))
existing_json_stems = {p.stem for p in JSON_OUT_DIR.glob("*.json")}

total_files = len(las_files)
new_json_count = 0
skipped_by_json = 0
failed_files: List[str] = []

job_start = time.perf_counter()

for idx, las_file in enumerate(las_files, start=1):
    file_stem = las_file.stem
    file_name = las_file.name

    print(f"\n[{idx}/{total_files}] 开始处理: {file_name}")

    if file_stem in existing_json_stems:
        skipped_by_json += 1
        print("  - 状态: 跳过（已存在 JSON）")
        continue

    las_text = las_file.read_text(encoding="utf-8", errors="ignore")
    print(f"  - 文件字符数: {len(las_text)}")

    prompt = build_prompt(file_name=file_name, las_text=las_text, schema_columns=columns)
    print(f"  - Prompt 字符数: {len(prompt)}")

    ok = False
    last_err = None
    well_start = time.perf_counter()

    for attempt in range(MAX_RETRY + 1):
        attempt_no = attempt + 1
        req_start = time.perf_counter()
        print(f"  - 第 {attempt_no} 次请求: 发送中...")
        try:
            result = call_llm_once(prompt)
            req_elapsed = time.perf_counter() - req_start
            print(f"  - 第 {attempt_no} 次请求: 完成，耗时 {req_elapsed:.2f}s")

            normalized = validate_and_normalize_result(
                result,
                file_name=file_name,
                schema_columns=columns,
                key_column=KEY_COLUMN,
            )
            print("  - JSON 判定: 合法")

            json_out = {
                "file_name": file_name,
                "file_stem": file_stem,
                "las_text_length": len(las_text),
                "classified": normalized,
                "llm_raw": result,
            }
            (JSON_OUT_DIR / f"{file_stem}.json").write_text(
                json.dumps(json_out, ensure_ascii=False, indent=2),
                encoding="utf-8",
            )

            print("  - 已写入单井 JSON")
            print("  - classified:")
            print(json.dumps(normalized, ensure_ascii=False, indent=2))

            new_json_count += 1
            ok = True
            break
        except Exception as exc:
            req_elapsed = time.perf_counter() - req_start
            last_err = exc
            print(f"  - 第 {attempt_no} 次请求: 失败，耗时 {req_elapsed:.2f}s")
            print(f"  - JSON 判定: 非法 / 请求失败，原因: {exc}")
            if attempt < MAX_RETRY:
                print("  - 将重试 1 次")
                time.sleep(2)

    well_elapsed = time.perf_counter() - well_start
    if ok:
        print(f"  - 井处理完成，总耗时 {well_elapsed:.2f}s")
    else:
        print(f"  - 井处理失败，总耗时 {well_elapsed:.2f}s")
        failed_files.append(f"{file_name}: {last_err}")

job_elapsed = time.perf_counter() - job_start

print("\n===== 运行汇总 =====")
print(f"总 LAS 文件: {total_files}")
print(f"已有 JSON 跳过: {skipped_by_json}")
print(f"本次新增 JSON: {new_json_count}")
print(f"失败数量: {len(failed_files)}")
print(f"总耗时: {job_elapsed:.2f}s")
if failed_files:
    print("失败样例（最多显示前10条）:")
    for x in failed_files[:10]:
        print("-", x)
print(f"单井 JSON 目录: {JSON_OUT_DIR}")

In [ ]:
# 第二步：从 JSON 组装 import.csv（可单独重复执行）
rows: List[Dict[str, str]] = []
json_files = sorted(JSON_OUT_DIR.glob("*.json"))

for jf in json_files:
    try:
        obj = json.loads(jf.read_text(encoding="utf-8"))
        classified = obj.get("classified", {})
        if not isinstance(classified, dict):
            continue

        row = {c: "" for c in columns}
        for c in columns:
            if c in classified:
                row[c] = normalize_value(classified[c])

        # 主键列兜底
        if not row.get(KEY_COLUMN):
            row[KEY_COLUMN] = obj.get("file_name", jf.stem)

        rows.append(row)
    except Exception:
        # 损坏 JSON 直接跳过，避免影响整体组装
        continue

out_df = pd.DataFrame(rows, columns=columns)
if not out_df.empty:
    out_df[KEY_COLUMN] = out_df[KEY_COLUMN].astype(str).str.strip()
    out_df = out_df.drop_duplicates(subset=[KEY_COLUMN], keep="last")

out_df.to_csv(OUT_CSV, index=False, encoding="utf-8-sig")

print(f"JSON 文件数: {len(json_files)}")
print(f"组装后行数: {len(out_df)}")
print(f"已导出: {OUT_CSV}")